In [ ]:
import os
os.environ["HF_USERNAME"] = "himanshunakrani9"
os.environ["HF_TOKEN"] = ""

In [ ]:
#!/usr/bin/env python3
"""
Generate 10 high-quality reasoning samples using vLLM server on localhost:8888
with Qwen/Qwen3.6-35B-A3B model and push to HuggingFace.
"""

import json
import os
import re
import random
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime

from openai import OpenAI
from huggingface_hub import HfApi, login
from tqdm import tqdm

# Configuration - can be set via environment variables
VLLM_BASE_URL = os.getenv("VLLM_BASE_URL", "http://localhost:30000/v1")
VLLM_MODEL = os.getenv("VLLM_MODEL", "Qwen/Qwen3.6-35B-A3B")
NUM_SAMPLES = 200 # int(os.getenv("NUM_SAMPLES", "2"))
CONCURRENCY = int(os.getenv("CONCURRENCY", "4"))   # parallel in-flight requests to SGLang/vLLM
MAX_TOKENS = int(os.getenv("MAX_TOKENS", "8192"))
ATTEMPT_CAP_MULT = int(os.getenv("ATTEMPT_CAP_MULT", "6"))  # total attempts = NUM_SAMPLES * this
HF_PRIVATE = os.getenv("HF_PRIVATE", "true").lower() in ("1", "true", "yes")  # default PRIVATE for safety

# Connect to vLLM server
client = OpenAI(
    base_url=VLLM_BASE_URL,
    api_key="dummy"  # vLLM doesn't require real API key
)

# ---------------------------------------------------------------------------
# Prompt engineering: elicit frontier-level (Opus 4.7 class) reasoning
# ---------------------------------------------------------------------------

SYSTEM_PROMPT = """You are a world-class problem designer and reasoner, operating at the level of the most advanced frontier models.

Your job is to produce ONE synthetic training example that exemplifies **elite multi-step reasoning**. The example must be:
- Novel (not a memorized textbook problem)
- Non-trivial (requires genuine multi-step deduction, not pattern matching)
- Rigorously correct (every claim and computation verified)
- Self-contained (all information needed is in the question)
- Pedagogically valuable (a strong solver would learn from the reasoning trace)

You must think like a careful scientist: decompose, hypothesize, compute, verify, reflect. Never hand-wave. Never fabricate facts.

Output MUST be a single valid JSON object. No prose outside the JSON. No markdown code fences."""

# Diverse topic seeds to force variety across samples
TOPIC_SEEDS = [
    ("mathematical_reasoning", "Olympiad-style combinatorics, number theory, or inequality problems that require a clever insight plus rigorous proof."),
    ("algorithmic_coding", "A non-obvious algorithmic problem (graphs, DP, invariants). Include a correct Python reference solution and complexity analysis."),
    ("logical_deduction", "A multi-constraint logic puzzle (knights/knaves, seating, scheduling) solvable only by systematic case analysis."),
    ("scientific_reasoning", "A physics, chemistry, or biology Fermi/estimation or derivation problem requiring unit analysis and first-principles thinking."),
    ("code_debugging", "A subtle bug in a realistic code snippet (off-by-one, race condition, numerical stability). Require root-cause analysis, not just a patch."),
    ("causal_inference", "A real-world scenario where naive correlation misleads; require confounders, counterfactuals, or Simpson's paradox reasoning."),
    ("game_theory", "A strategic interaction requiring backward induction, mixed strategies, or mechanism design reasoning."),
    ("probabilistic_reasoning", "A Bayesian update, Monty-Hall-style, or martingale problem where intuition commonly fails."),
    ("multi_hop_inference", "A problem requiring chaining 4+ independent facts or constraints to reach a conclusion."),
    ("proof_construction", "Ask for a proof (by induction, contradiction, or construction) of a non-obvious statement."),
]

DIFFICULTY_LEVELS = [
    "graduate-level (top 1% of STEM graduates would find it challenging)",
    "research-level (requires synthesizing multiple advanced concepts)",
    "competition-level (IMO / Putnam / ICPC caliber)",
]

# Plain string with <<PLACEHOLDERS>> + str.replace() — avoids both .format()'s
# curly-brace interpretation and string.Template's $-interpretation (our body
# legitimately contains both { } and $ as example bad content).
USER_PROMPT_TEMPLATE = """Generate ONE elite reasoning training example.

**Topic**: <<TOPIC_NAME>> — <<TOPIC_DESC>>
**Difficulty**: <<DIFFICULTY>>
**Random seed (for novelty)**: <<SEED>>

## Required JSON schema
{
  "topic": "<<TOPIC_NAME>>",
  "difficulty": "<<DIFFICULTY>>",
  "question": "A fully self-contained problem statement. State every assumption explicitly. If code or data is involved, include it verbatim.",
  "reasoning": "A long-form, Opus-class chain-of-thought that follows the PHASES below. Use explicit section headers. Be exhaustive, not concise.",
  "verification": "An INDEPENDENT check of the answer (alternative method, sanity bounds, plug-back, or counter-example search). Must actually confirm correctness, not restate the solution.",
  "answer": "The final answer, stated crisply and unambiguously. For numerical answers include units. For proofs, state the theorem being proved.",
  "pitfalls": ["list of 2-4 common wrong approaches or traps a weaker solver would fall into, with a sentence explaining why each fails"]
}

## Required reasoning PHASES
The `reasoning` string MUST contain exactly these six headers, each on its own line, spelled EXACTLY as shown (case-sensitive, including the `###` and the header word):

### Understand
(Restate the problem, identify knowns/unknowns, flag ambiguities and resolve them.)

### Decompose
(Break the problem into sub-problems or cases. State the high-level plan.)

### Explore
(Consider at least TWO candidate approaches. Weigh tradeoffs before committing.)

### Execute
(Carry out the chosen approach with every step justified. Show all algebra / code / logic.)

### Verify
(Cross-check via a DIFFERENT method: numerical sanity, symmetry, limiting case, alternative derivation.)

### Reflect
(What was the key insight? Under what perturbation would the answer change?)

## Quality bar (non-negotiable)
- No arithmetic or logical errors — recompute anything you're unsure of.
- No hand-waving ("clearly", "obviously") for non-trivial steps.
- Reasoning length should match difficulty — expect 400-1200 words for the `reasoning` field.
- The `verification` step MUST use a genuinely different method than `Execute`.
- Return ONLY the JSON object. No preamble, no postscript, no code fences.

## JSON FORMATTING RULES (strictly enforced — violations cause the sample to be discarded)
- **NO LaTeX inside any string value.** Do NOT write `\\mu`, `\\frac`, `\\Delta`, `\\,`, `\\text{...}`, `$...$`, etc. Use plain English instead: `mu`, `Delta`, `integral of`, `square root of`, etc. Math should be expressed in readable ASCII like `x^2`, `sqrt(2)`, `sum_{i=1}^{n} i`, `integral_0^pi sin(x) dx`.
- **NO unescaped double quotes inside string values.** If you need to quote something inside a value, use single quotes `'like this'` or backticks `` `like this` ``. Never `"like this"` inside another string.
- **Escape all backslashes.** The only valid escapes are `\\n`, `\\t`, `\\"`, `\\\\`. If you need a literal backslash, write `\\\\`.
- **All 6 phase headers (### Understand, ### Decompose, ### Explore, ### Execute, ### Verify, ### Reflect) MUST appear inside the `reasoning` string, spelled exactly.** Do not stop after 2 or 3 phases. Complete all six before closing the string.
- **The top-level `verification` field is MANDATORY and SEPARATE from the `### Verify` phase inside `reasoning`.** Even though you'll write a `### Verify` section in the reasoning trace, you MUST ALSO emit a distinct top-level `"verification": "..."` field in the JSON that summarizes the independent check in 1-3 sentences. Do NOT omit this field. Samples without a top-level `verification` field are discarded.
"""


def build_user_prompt():
    topic_name, topic_desc = random.choice(TOPIC_SEEDS)
    difficulty = random.choice(DIFFICULTY_LEVELS)
    seed = random.randint(10**6, 10**9)
    prompt = (USER_PROMPT_TEMPLATE
              .replace("<<TOPIC_NAME>>", topic_name)
              .replace("<<TOPIC_DESC>>", topic_desc)
              .replace("<<DIFFICULTY>>", difficulty)
              .replace("<<SEED>>", str(seed)))
    return prompt, topic_name, difficulty


# JSON's only valid backslash-escape tails: "\/bfnrtu. Everything else is illegal.
VALID_JSON_ESCAPE_CHARS = set('"\\/bfnrtu')


def _sanitize_json_string_literals(text: str) -> str:
    """Walk text as JSON-ish; inside string literals, fix raw newlines/tabs AND
    escape invalid LaTeX-style backslash escapes (\\mu -> \\\\mu, etc.).
    Leaves content outside strings untouched."""
    out = []
    in_str = False
    i = 0
    n = len(text)
    while i < n:
        ch = text[i]
        if in_str:
            if ch == "\\" and i + 1 < n:
                nxt = text[i + 1]
                if nxt in VALID_JSON_ESCAPE_CHARS:
                    # Valid JSON escape: keep both chars
                    out.append(ch); out.append(nxt); i += 2; continue
                else:
                    # Invalid escape (e.g. \mu, \,, \{): double the backslash so it
                    # survives as a literal backslash inside the string
                    out.append("\\\\"); out.append(nxt); i += 2; continue
            elif ch == '"':
                out.append(ch); in_str = False; i += 1; continue
            elif ch == "\n":
                out.append("\\n"); i += 1; continue
            elif ch == "\r":
                out.append("\\r"); i += 1; continue
            elif ch == "\t":
                out.append("\\t"); i += 1; continue
            else:
                out.append(ch); i += 1; continue
        else:
            if ch == '"':
                in_str = True
            out.append(ch); i += 1
    return "".join(out)


def _try_loads_lenient(candidate: str):
    """Try json.loads, with escalating lenient fixes for common model mistakes."""
    # 1. Strict
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        pass
    # 2. Normalize smart quotes
    fixed = (candidate
             .replace("\u201c", '"').replace("\u201d", '"')
             .replace("\u2018", "'").replace("\u2019", "'"))
    try:
        return json.loads(fixed)
    except json.JSONDecodeError:
        pass
    # 3. Fix raw newlines AND invalid LaTeX-style backslash escapes inside strings
    sanitized = _sanitize_json_string_literals(fixed)
    try:
        return json.loads(sanitized)
    except json.JSONDecodeError:
        pass
    # 4. Strip trailing commas before } or ]
    no_trailing = re.sub(r",(\s*[}\]])", r"\1", sanitized)
    try:
        return json.loads(no_trailing)
    except json.JSONDecodeError:
        return None


# -------- Schema-aware fallback --------
# When the model emits unescaped inner quotes (or other subtly broken JSON), strict
# parsing is impossible to recover generically. Since WE control the schema, extract
# fields by name: look for "key": "value" where value ends just before the next known
# key. This rescues a large class of otherwise-dropped samples.
SCHEMA_KEYS = ["topic", "difficulty", "question", "reasoning", "verification", "answer"]


def schema_aware_extract(text: str):
    """Regex-extract known fields. Returns dict with at least question/reasoning/verification/answer on success, else None."""
    # Strip thinking and code fences (same as extract_json)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL)

    result = {}
    # For each key, find "key" : "..." where the closing " is followed by
    # a comma and the next known key (or } ) -- this defines the end of value.
    next_keys_alt = "|".join(re.escape(k) for k in SCHEMA_KEYS + ["pitfalls"])
    for key in SCHEMA_KEYS:
        # Non-greedy capture until we see ",\s*"<another known key>" OR closing bracket
        pat = re.compile(
            r'"' + re.escape(key) + r'"\s*:\s*"(.*?)"\s*(?:,\s*"(?:' + next_keys_alt + r')"|[}\]])',
            re.DOTALL,
        )
        m = pat.search(text)
        if m:
            result[key] = m.group(1)
    # Accept if at least 3 of 4 load-bearing fields parsed; quality gate will reject
    # on missing fields with an informative reason rather than opaque "unparsable".
    required_core = {"question", "reasoning", "answer"}
    if required_core.issubset(result.keys()):
        return result
    return None


def extract_json(text: str):
    """Robustly extract the first top-level JSON object from model output."""
    if not text:
        return None
    original_text = text

    # Strip Qwen3-style <think>...</think> blocks (their braces confuse parsing)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    # Also strip unterminated <think> (happens on truncation)
    text = re.sub(r"<think>.*", "", text, flags=re.DOTALL)

    # Strip common wrappers: ```json ... ``` or ``` ... ```
    fence = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
    if fence:
        parsed = _try_loads_lenient(fence.group(1))
        if parsed is not None:
            return parsed

    # Brace-matching scan for the first balanced JSON object
    start = text.find("{")
    while start != -1:
        depth = 0
        in_str = False
        escape = False
        for i in range(start, len(text)):
            ch = text[i]
            if in_str:
                if escape:
                    escape = False
                elif ch == "\\":
                    escape = True
                elif ch == '"':
                    in_str = False
            else:
                if ch == '"':
                    in_str = True
                elif ch == "{":
                    depth += 1
                elif ch == "}":
                    depth -= 1
                    if depth == 0:
                        candidate = text[start:i + 1]
                        parsed = _try_loads_lenient(candidate)
                        if parsed is not None:
                            return parsed
                        break
        start = text.find("{", start + 1)
    # Last resort: schema-aware field extraction
    return schema_aware_extract(original_text)


REQUIRED_FIELDS = {"question", "reasoning", "verification", "answer"}
DEBUG_DIR = "data/qwen_samples/_debug"
_debug_saved = 0
DEBUG_MAX = 5  # save at most this many raw failures for inspection


_VERIFY_SECTION_RE = re.compile(
    r"###\s*Verify\s*\n(.*?)(?=\n###\s*(?:Reflect|Execute|Understand|Decompose|Explore)|\Z)",
    re.DOTALL | re.IGNORECASE,
)


def _recover_verification_from_reasoning(sample: dict) -> bool:
    """If the model put its verification content inside reasoning's '### Verify'
    section but omitted the top-level field, lift it out. Returns True if rescued."""
    reasoning = sample.get("reasoning", "")
    if not isinstance(reasoning, str):
        return False
    m = _VERIFY_SECTION_RE.search(reasoning)
    if not m:
        return False
    extracted = m.group(1).strip()
    if len(extracted) < 40:  # too short to be useful
        return False
    sample["verification"] = extracted
    return True


def quality_reason(sample) -> str:
    """Return '' if sample is acceptable, else a short reason string."""
    if not isinstance(sample, dict):
        return "not_a_dict"
    # Rescue: if verification is missing but the reasoning has a '### Verify' section,
    # lift it out. This handles the #1 observed failure mode where the model emits
    # ### Verify inside reasoning but forgets the top-level `verification` field.
    if not sample.get("verification"):
        _recover_verification_from_reasoning(sample)
    missing = REQUIRED_FIELDS - set(sample.keys())
    if missing:
        return f"missing_fields={sorted(missing)}"
    reasoning = sample.get("reasoning", "")
    if not isinstance(reasoning, str):
        return "reasoning_not_str"
    if len(reasoning) < 400:
        return f"reasoning_too_short({len(reasoning)}<400)"
    phases = ["Understand", "Decompose", "Explore", "Execute", "Verify", "Reflect"]
    hits = [p for p in phases if p.lower() in reasoning.lower()]
    # Must contain Execute + Verify (the load-bearing phases) plus at least one earlier phase
    must_have = {"Execute", "Verify"}
    if not must_have.issubset(set(hits)):
        return f"missing_core_phases(missing={sorted(must_have - set(hits))}, found={hits})"
    if len(hits) < 3:
        return f"too_few_phase_headers({len(hits)}/6 found={hits})"
    if not sample.get("answer"):
        return "empty_answer"
    if not sample.get("verification"):
        return "empty_verification"
    return ""


def _dump_debug(tag: str, raw: str, attempt: int):
    """Save raw output of the first few failures to disk for inspection."""
    global _debug_saved
    if _debug_saved >= DEBUG_MAX:
        return
    os.makedirs(DEBUG_DIR, exist_ok=True)
    path = os.path.join(DEBUG_DIR, f"fail_{_debug_saved:02d}_{tag}_attempt{attempt}.txt")
    try:
        with open(path, "w") as f:
            f.write(raw)
        _debug_saved += 1
        print(f"    (raw output saved to {path})")
    except Exception:
        pass


def generate_sample(max_attempts: int = 3):
    """Generate one high-quality reasoning sample with validation and retry."""
    last_error = None
    for attempt in range(1, max_attempts + 1):
        user_prompt, topic, difficulty = build_user_prompt()
        try:
            response = client.chat.completions.create(
                model=VLLM_MODEL,
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt},
                ],
                temperature=0.7,
                top_p=0.95,
                max_tokens=MAX_TOKENS,
                timeout=600,
                # Disable Qwen3 thinking mode so reasoning lands in the JSON field,
                # not a <think> block we'd have to strip. Harmless for non-Qwen models.
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            )
            generated = response.choices[0].message.content or ""
            finish_reason = response.choices[0].finish_reason
            data = extract_json(generated)
            if data is None:
                last_error = f"json_parse_failed (finish={finish_reason}, len={len(generated)})"
                _dump_debug("unparsable", generated, attempt)
                continue
            reason = quality_reason(data)
            if reason:
                last_error = f"quality:{reason} (finish={finish_reason})"
                _dump_debug("lowquality", generated, attempt)
                continue
            # Ensure metadata is present even if the model omitted it
            data.setdefault("topic", topic)
            data.setdefault("difficulty", difficulty)
            return data
        except Exception as e:
            last_error = f"api_error:{e!r}"
            time.sleep(min(2 ** attempt, 10))
    print(f"  ↳ sample rejected after {max_attempts} attempts (last_error={last_error})")
    return None

def preflight_check():
    """Verify the configured endpoint is actually an OpenAI-compatible server."""
    import urllib.request
    import urllib.error
    url = VLLM_BASE_URL.rstrip("/") + "/models"
    try:
        with urllib.request.urlopen(url, timeout=10) as resp:
            ctype = resp.headers.get("Content-Type", "")
            body = resp.read(4096).decode("utf-8", errors="replace")
            if "application/json" not in ctype:
                print(f"ERROR: {url} returned Content-Type={ctype!r}, not JSON.")
                print(f"  First bytes: {body[:200]!r}")
                print("  This is not a vLLM / OpenAI-compatible endpoint.")
                print("  Check your VLLM_BASE_URL (you may be hitting Jupyter or another service).")
                return False
            data = json.loads(body)
            models = [m.get("id") for m in data.get("data", [])]
            print(f"  Endpoint OK. Available models: {models}")
            if VLLM_MODEL not in models and models:
                print(f"  WARNING: VLLM_MODEL={VLLM_MODEL!r} not in server's model list.")
                print(f"  Consider: export VLLM_MODEL={models[0]!r}")
            return True
    except urllib.error.HTTPError as e:
        print(f"ERROR: HTTP {e.code} from {url}. Server is not an OpenAI-compatible endpoint.")
        return False
    except Exception as e:
        print(f"ERROR: cannot reach {url}: {e}")
        return False


def main():
    print("=" * 60)
    print("Generating High-Quality Synthetic Reasoning Data")
    print("=" * 60)
    print(f"Model:         {VLLM_MODEL}")
    print(f"Server:        {VLLM_BASE_URL}")
    print(f"Target:        {NUM_SAMPLES} samples")
    print(f"Concurrency:   {CONCURRENCY} in-flight requests")
    print(f"Max tokens:    {MAX_TOKENS}")
    print("=" * 60)

    print("\n[preflight] Checking endpoint...")
    if not preflight_check():
        print("\nAborting. Fix VLLM_BASE_URL and retry.")
        return

    # Create data directory + output file (append-as-we-go for crash safety)
    os.makedirs("data/qwen_samples", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    local_file = f"data/qwen_samples/samples_{timestamp}.jsonl"
    print(f"\nIncremental output file: {local_file}")

    samples = []
    max_total_attempts = NUM_SAMPLES * ATTEMPT_CAP_MULT
    attempts_done = 0
    file_lock = threading.Lock()
    start_time = time.time()

    print(f"\nGenerating with {CONCURRENCY} parallel workers (cap {max_total_attempts} total attempts)...\n")
    print("(Press Ctrl-C once for graceful shutdown — in-flight requests finish, then save & upload what's done.)\n")
    pbar = tqdm(total=NUM_SAMPLES, desc="High-quality samples")

    interrupted = {"flag": False}  # mutable holder so closure can set it
    pool = ThreadPoolExecutor(max_workers=CONCURRENCY)
    in_flight = set()

    def submit_one():
        nonlocal attempts_done
        attempts_done += 1
        in_flight.add(pool.submit(generate_sample))

    try:
        # Prime the pool
        for _ in range(min(CONCURRENCY, max_total_attempts)):
            submit_one()

        while in_flight and len(samples) < NUM_SAMPLES and not interrupted["flag"]:
            # Wait for any one future to finish, then re-prime
            done_set = set()
            for fut in as_completed(list(in_flight), timeout=None):
                done_set.add(fut)
                break
            for fut in done_set:
                in_flight.discard(fut)
                try:
                    sample = fut.result()
                except Exception as e:
                    sample = None
                    print(f"  worker exception: {e!r}")
                if sample and len(samples) < NUM_SAMPLES:
                    samples.append(sample)
                    with file_lock:
                        with open(local_file, "a") as f:
                            f.write(json.dumps(sample, ensure_ascii=False) + "\n")
                    elapsed = time.time() - start_time
                    rate = len(samples) / max(elapsed / 60.0, 1e-9)
                    pbar.update(1)
                    pbar.set_postfix(
                        topic=sample.get("topic", "?")[:16],
                        rate=f"{rate:.2f}/min",
                        attempts=attempts_done,
                    )
            # Re-prime if we still need more and haven't exceeded cap
            while (not interrupted["flag"]
                   and len(in_flight) < CONCURRENCY
                   and len(samples) + len(in_flight) < NUM_SAMPLES
                   and attempts_done < max_total_attempts):
                submit_one()
    except KeyboardInterrupt:
        interrupted["flag"] = True
        print("\n\n[!] KeyboardInterrupt received — stopping submission of new work.")
        print(f"    Waiting on {len(in_flight)} in-flight request(s) to finish (up to ~10 min each)...")
        print("    Press Ctrl-C again to abandon in-flight work (their outputs will be lost).")
        # Drain in-flight futures (but allow second Ctrl-C to hard-abort)
        try:
            for fut in as_completed(list(in_flight), timeout=None):
                try:
                    sample = fut.result()
                    if sample and len(samples) < NUM_SAMPLES:
                        samples.append(sample)
                        with file_lock:
                            with open(local_file, "a") as f:
                                f.write(json.dumps(sample, ensure_ascii=False) + "\n")
                        pbar.update(1)
                except Exception as e:
                    print(f"    in-flight worker error: {e!r}")
        except KeyboardInterrupt:
            print("\n[!] Second Ctrl-C — abandoning in-flight work.")
    finally:
        try:
            # Python 3.9+: cancel_futures cancels queued-but-not-started work
            pool.shutdown(wait=False, cancel_futures=True)
        except TypeError:
            pool.shutdown(wait=False)

    pbar.close()
    elapsed = time.time() - start_time
    print(f"\n  attempts used: {attempts_done} / cap {max_total_attempts}")
    print(f"  wall time:     {elapsed/60:.1f} min")
    print(f"  success rate:  {len(samples)}/{attempts_done} = {100*len(samples)/max(attempts_done,1):.1f}%")

    print(f"\n{'=' * 60}")
    print(f"Generation complete! Generated {len(samples)} valid samples.")
    print(f"{'=' * 60}")

    # Short-circuit: nothing to save or upload
    if not samples:
        print("\nNo samples generated — skipping save and HuggingFace upload.")
        print("Inspect data/qwen_samples/_debug/*.txt for raw model output to diagnose.")
        return

    print(f"Saved incrementally to: {local_file}")
    
    # Preview first sample
    if samples:
        print("\n" + "=" * 60)
        print("Sample Preview:")
        print("=" * 60)
        print(json.dumps(samples[0], indent=2, ensure_ascii=False))
    
    # Push to HuggingFace
    print("\n" + "=" * 60)
    print("Pushing to HuggingFace")
    print("=" * 60)
    
    hf_username = os.getenv("HF_USERNAME")
    hf_token = os.getenv("HF_TOKEN")
    
    if not hf_username:
        print("ERROR: HF_USERNAME environment variable not set")
        print("Please set it with: export HF_USERNAME=your_username")
        return
    
    if not hf_token:
        print("ERROR: HF_TOKEN environment variable not set")
        print("Please set it with: export HF_TOKEN=your_token")
        return
    
    # Login to HuggingFace
    login(token=hf_token)
    
    # Create dataset repo name
    dataset_name = f"{hf_username}/qwen-reasoning-samples-{timestamp}"
    
    # Upload dataset
    api = HfApi()
    
    print(f"\nCreating/updating dataset: {dataset_name}  (private={HF_PRIVATE})")
    api.create_repo(
        repo_id=dataset_name,
        repo_type="dataset",
        exist_ok=True,
        private=HF_PRIVATE,
    )
    
    # Upload the samples file
    api.upload_file(
        path_or_fileobj=local_file,
        path_in_repo="samples.jsonl",
        repo_id=dataset_name,
        repo_type="dataset"
    )
    
    # Add dataset card
    dataset_card = f"""---
license: mit
task_categories:
- text-generation
- question-answering
language:
- en
tags:
- reasoning
- chain-of-thought
- synthetic
- frontier-reasoning
---

# Frontier-Class Synthetic Reasoning Samples

## Dataset Description
{len(samples)} synthetic reasoning examples generated with **{VLLM_MODEL}** via vLLM, using a structured prompt designed to elicit **frontier-level (Opus 4.7 class) multi-phase reasoning**.

Each example is gated through a quality filter that requires the reasoning trace to follow an explicit 6-phase structure (Understand → Decompose → Explore → Execute → Verify → Reflect) and to include an independent verification step.

## Generation Details
- **Model**: {VLLM_MODEL}
- **Server**: vLLM at {VLLM_BASE_URL}
- **Generation Date**: {timestamp}
- **Temperature**: 0.7, **top-p**: 0.95, **max_tokens**: {MAX_TOKENS}
- **Topic diversity**: rotated across math, algorithms, logic, science, debugging, causal inference, game theory, probability, multi-hop, proofs
- **Difficulty**: graduate / research / competition level

## Dataset Schema
Each JSONL row contains:
- `topic`: category label
- `difficulty`: difficulty tier
- `question`: self-contained problem statement
- `reasoning`: long-form chain-of-thought with 6 labeled phases
- `verification`: INDEPENDENT check of the answer using a different method
- `answer`: final crisp answer
- `pitfalls`: 2-4 common wrong approaches with explanations

## Usage
```python
from datasets import load_dataset

dataset = load_dataset("{dataset_name}")
print(dataset["train"][0]["reasoning"])
```

## Intended Use
SFT / distillation data for training smaller models to imitate structured, verified reasoning traces.
"""
    
    api.upload_file(
        path_or_fileobj=dataset_card.encode('utf-8'),
        path_in_repo="README.md",
        repo_id=dataset_name,
        repo_type="dataset"
    )
    
    print(f"\n{'=' * 60}")
    print("SUCCESS!")
    print(f"{'=' * 60}")
    print(f"Dataset pushed to: https://huggingface.co/datasets/{dataset_name}")
    print(f"Total samples: {len(samples)}")
    print(f"{'=' * 60}")

if __name__ == "__main__":
    main()


Generating High-Quality Synthetic Reasoning Data
Model:         Qwen/Qwen3.6-35B-A3B
Server:        http://localhost:30000/v1
Target:        200 samples
Concurrency:   4 in-flight requests
Max tokens:    8192

[preflight] Checking endpoint...
  Endpoint OK. Available models: ['Qwen/Qwen3.6-35B-A3B']

Incremental output file: data/qwen_samples/samples_20260421_223122.jsonl

Generating with 4 parallel workers (cap 1200 total attempts)...

(Press Ctrl-C once for graceful shutdown — in-flight requests finish, then save & upload what's done.)



High-quality samples:   1% 2/200 [00:40<58:08, 17.62s/it, attempts=5, rate=2.94/min, topic=multi_hop_infere]  

    (raw output saved to data/qwen_samples/_debug/fail_00_lowquality_attempt1.txt)
    (raw output saved to data/qwen_samples/_debug/fail_01_lowquality_attempt1.txt)


High-quality samples:   2% 3/200 [01:17<1:26:33, 26.36s/it, attempts=6, rate=2.32/min, topic=algorithmic_codi]

    (raw output saved to data/qwen_samples/_debug/fail_02_lowquality_attempt2.txt)


High-quality samples:   2% 5/200 [01:51<1:01:25, 18.90s/it, attempts=8, rate=2.70/min, topic=scientific_reaso]

    (raw output saved to data/qwen_samples/_debug/fail_03_unparsable_attempt2.txt)
    (raw output saved to data/qwen_samples/_debug/fail_04_lowquality_attempt1.txt)


High-quality samples:   3% 6/200 [02:45<1:39:55, 30.90s/it, attempts=9, rate=2.18/min, topic=proof_constructi]